# Experiment Evaluation Tables

## Goal

Find every `pt_experiment.csv` below the project's `outputs` directory and display one table per experiment. Every result row is retained, while only run identifiers, selected retrieval metrics, and their related statistical columns are shown.

## Setup

Edit `SELECTED_METRICS` to change the displayed measures. When `INCLUDE_RELATED_COLUMNS` is `True`, columns such as `ndcg_cut_10 p-value`, `ndcg_cut_10 reject`, `ndcg_cut_10 +`, and `ndcg_cut_10 -` are included automatically when present.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

# Project evaluation metrics. Add or remove names here as needed.
SELECTED_METRICS = [
    "ndcg_cut_10",
    "AP(rel=2)",
    "RR(rel=2)",
    "RR@10",
    "recall_100",
]
IDENTIFIER_COLUMNS = ["name", "run", "system", "model", "model_id", "dimension"]
INCLUDE_RELATED_COLUMNS = True
EVALUATION_FILENAME = "pt_experiment.csv"

## Discover evaluation files

The root lookup works whether the notebook is started from the repository root or from `analysis_viz`.

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the nearest parent containing the codebase directory."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "matryoshka_optimization_codebase").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from inside the Master-Thesis repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUTS_DIR = PROJECT_ROOT / "matryoshka_optimization_codebase" / "outputs"
evaluation_files = sorted(OUTPUTS_DIR.rglob(EVALUATION_FILENAME)) if OUTPUTS_DIR.exists() else []

print(f"Outputs directory: {OUTPUTS_DIR}")
print(f"Found {len(evaluation_files)} evaluation file(s).")
for evaluation_file in evaluation_files:
    print(" -", evaluation_file.relative_to(OUTPUTS_DIR))

## Results

Each CSV is loaded independently. The experiment label is its directory path relative to `outputs`, so nested experiments and repeated folder names remain distinguishable. A canonical column list is built across all files and applied to every table, guaranteeing identical columns in identical order; unavailable values appear as `NaN`.

In [ ]:
def build_canonical_columns(column_sets: list[list[str]]) -> list[str]:
    """Build one deterministic display schema shared by every experiment table."""
    available_columns = {column for columns in column_sets for column in columns}
    canonical_columns = [column for column in IDENTIFIER_COLUMNS if column in available_columns]

    for metric in SELECTED_METRICS:
        if metric in available_columns:
            canonical_columns.append(metric)
        if INCLUDE_RELATED_COLUMNS:
            related_columns = sorted(
                column for column in available_columns if column.startswith(f"{metric} ")
            )
            canonical_columns.extend(related_columns)

    return list(dict.fromkeys(canonical_columns))


# First load every file so one common column schema can be computed.
loaded_results: dict[str, pd.DataFrame] = {}
load_errors: dict[str, str] = {}
for evaluation_file in evaluation_files:
    experiment = evaluation_file.parent.relative_to(OUTPUTS_DIR).as_posix()
    try:
        loaded_results[experiment] = pd.read_csv(evaluation_file)
    except Exception as exc:
        load_errors[experiment] = str(exc)

canonical_columns = build_canonical_columns(
    [results.columns.tolist() for results in loaded_results.values()]
)
results_by_experiment: dict[str, pd.DataFrame] = {}
discovery_rows = []

if not evaluation_files:
    display(Markdown(f"> No `{EVALUATION_FILENAME}` files were found below `{OUTPUTS_DIR}`."))
else:
    print("Canonical column order:", canonical_columns)
    for evaluation_file in evaluation_files:
        experiment = evaluation_file.parent.relative_to(OUTPUTS_DIR).as_posix()
        display(Markdown(f"### `{experiment}`"))

        if experiment in load_errors:
            error = load_errors[experiment]
            discovery_rows.append(
                {
                    "experiment": experiment,
                    "rows": None,
                    "displayed_columns": ", ".join(canonical_columns) or "(none)",
                    "missing_metrics": "unknown",
                    "status": f"read error: {error}",
                }
            )
            display(Markdown(f"> Could not read `{evaluation_file.name}`: `{error}`"))
            continue

        full_results = loaded_results[experiment]
        # reindex enforces the same columns and order; absent columns become NaN.
        selected_results = full_results.reindex(columns=canonical_columns).copy()
        results_by_experiment[experiment] = selected_results
        missing_metrics = [metric for metric in SELECTED_METRICS if metric not in full_results.columns]
        discovery_rows.append(
            {
                "experiment": experiment,
                "rows": len(full_results),
                "displayed_columns": ", ".join(canonical_columns) or "(none)",
                "missing_metrics": ", ".join(missing_metrics) or "(none)",
                "status": "ok" if canonical_columns else "no selected columns found",
            }
        )

        if canonical_columns:
            display(selected_results)
        else:
            display(Markdown("> None of the configured identifier or metric columns exists in any file."))
            print("Available columns:", full_results.columns.tolist())

## Checks

This compact summary confirms the number of rows retained from each experiment and reports missing configured metrics or unreadable files.

In [ ]:
discovery_summary = pd.DataFrame(
    discovery_rows,
    columns=["experiment", "rows", "displayed_columns", "missing_metrics", "status"],
)

if discovery_summary.empty:
    print("Nothing to validate until evaluation files are available.")
else:
    display(discovery_summary)